# Lunar Tear Patcher

Patch a NieR Re\[in\]carnation APK and master data for use with a private server.

**How to use:**
1. Fill in the **Configuration** form below.
2. Run cells top-to-bottom, or use **Runtime → Run all**.
3. Patched files will download to your browser.

**Two modes:**
- **Drive mode** (default): Check **use_gdrive**, enter paths relative to `/content/drive/` (e.g. `MyDrive/original.apk`).
- **URL mode**: Uncheck **use_gdrive**, paste download URLs into `apk_source` / `masterdata_source`.

Each section is independent — skip what you don't need.

In [ ]:
#@title Configuration

#@markdown ### Source Mode
use_gdrive = True #@param {type:"boolean"}

#@markdown ---
#@markdown ### File Sources
#@markdown With **use_gdrive** checked, enter Drive paths relative to `/content/drive/`
#@markdown (e.g. `MyDrive/original.apk`). Otherwise, enter download URLs.
apk_source = "" #@param {type:"string"}
masterdata_source = "" #@param {type:"string"}

#@markdown ---
#@markdown ### Server Settings
grpc_addr = "10.0.2.2:443" #@param {type:"string"}
http_addr = "10.0.2.2:8080" #@param {type:"string"}
auth_host = "" #@param {type:"string"}
scripts_branch = "main" #@param {type:"string"}

import os

if use_gdrive:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted at /content/drive\n')
    for label, src in [('APK', apk_source), ('Master data', masterdata_source)]:
        if not src:
            print(f'  {label}: (not set, will skip)')
            continue
        path = os.path.join('/content/drive', src)
        if os.path.isfile(path):
            size = os.path.getsize(path)
            print(f'  {label}: {path} — OK ({size / 1024:.1f} KB)')
        else:
            print(f'  {label}: {path} — NOT FOUND')
else:
    print('URL mode — files will be downloaded during patching.')

In [ ]:
#@title Install dependencies
import shutil, os

if not os.path.isdir('/content/scripts'):
    !git clone --depth 1 --branch {scripts_branch} https://gitlab.com/walter-sparrow-group/lunar-scripts.git /content/scripts
else:
    !cd /content/scripts && git fetch origin {scripts_branch} && git checkout {scripts_branch} && git pull origin {scripts_branch}
    print(f'Scripts updated to branch: {scripts_branch}')

!pip install -q pycryptodome msgpack lz4

!apt-get -qq update > /dev/null 2>&1

# apktool — install from bitbucket for a recent version
if not shutil.which('apktool'):
    !wget -q -O /usr/local/bin/apktool.jar \
      https://github.com/iBotPeaches/Apktool/releases/download/v2.11.1/apktool_2.11.1.jar
    !wget -q -O /usr/local/bin/apktool \
      https://raw.githubusercontent.com/iBotPeaches/Apktool/master/scripts/linux/apktool
    !chmod +x /usr/local/bin/apktool

# zipalign + apksigner
if not shutil.which('zipalign') or not shutil.which('apksigner'):
    !apt-get -qq install -y zipalign apksigner > /dev/null 2>&1

for tool in ['apktool', 'zipalign', 'apksigner', 'keytool']:
    path = shutil.which(tool)
    status = path or 'NOT FOUND'
    print(f'  {tool}: {status}')

print('\nDone.')

In [ ]:
#@title Patch APK
import urllib.request, shutil, os
from google.colab import files

assert apk_source, 'Set apk_source in the Configuration cell first.'
assert grpc_addr, 'Set grpc_addr in the Configuration cell first.'
assert http_addr, 'Set http_addr in the Configuration cell first.'

if use_gdrive:
    apk_path = os.path.join('/content/drive', apk_source)
    assert os.path.isfile(apk_path), f'File not found: {apk_path}'
    shutil.copy2(apk_path, '/content/original.apk')
    size_mb = os.path.getsize('/content/original.apk') / 1024 / 1024
    print(f'Copied APK from Drive: {apk_path} ({size_mb:.1f} MB)')
else:
    print(f'Downloading APK from {apk_source} ...')
    urllib.request.urlretrieve(apk_source, '/content/original.apk')
    size_mb = os.path.getsize('/content/original.apk') / 1024 / 1024
    print(f'  Downloaded {size_mb:.1f} MB')

print('\nDecompiling ...')
!apktool d /content/original.apk -o /content/patched -f

auth_flag = f'--auth-host {auth_host}' if auth_host else ''
print('\nPatching ...')
!python /content/scripts/android/patch_apk.py /content/patched --grpc-addr {grpc_addr} --http-addr {http_addr} {auth_flag}

print('\nRebuilding ...')
!apktool b /content/patched -o /content/patched_unaligned.apk

print('\nAligning ...')
!zipalign -p -f 4 /content/patched_unaligned.apk /content/patched.apk

if not os.path.exists('/content/debug.keystore'):
    print('\nGenerating debug keystore ...')
    !keytool -genkeypair -keystore /content/debug.keystore \
      -alias key -storepass android -keypass android \
      -keyalg RSA -keysize 2048 -validity 10000 \
      -dname "CN=Debug" 2>/dev/null

print('\nSigning ...')
!apksigner sign --ks /content/debug.keystore --ks-pass pass:android /content/patched.apk

size_mb = os.path.getsize('/content/patched.apk') / 1024 / 1024
print(f'\nDone! Patched APK: {size_mb:.1f} MB')

files.download('/content/patched.apk')

In [ ]:
#@title Patch Master Data
import urllib.request, shutil, os
from google.colab import files

assert masterdata_source, 'Set masterdata_source in the Configuration cell first.'

if use_gdrive:
    md_path = os.path.join('/content/drive', masterdata_source)
    assert os.path.isfile(md_path), f'File not found: {md_path}'
    shutil.copy2(md_path, '/content/masterdata.bin.e')
    size_kb = os.path.getsize('/content/masterdata.bin.e') / 1024
    print(f'Copied master data from Drive: {md_path} ({size_kb:.1f} KB)')
else:
    print(f'Downloading master data from {masterdata_source} ...')
    urllib.request.urlretrieve(masterdata_source, '/content/masterdata.bin.e')
    size_kb = os.path.getsize('/content/masterdata.bin.e') / 1024
    print(f'  Downloaded {size_kb:.1f} KB')

print('\nPatching ...')
!python /content/scripts/patch_masterdata.py \
  --input /content/masterdata.bin.e \
  --output /content/20240404193219.bin.e

size_kb = os.path.getsize('/content/20240404193219.bin.e') / 1024
print(f'\nDone! Patched master data: {size_kb:.1f} KB')

files.download('/content/20240404193219.bin.e')